# Laboratorium 2B · Fine-tuning YOLO na BDD100K

YOLOv5s startuje z wag COCO i uczy się pięciu klas: znak drogowy oraz sygnalizator czerwony, żółty, zielony lub z kolorem nieokreślonym (`none` w BDD100K).

Trenujemy jeden model na **15 000 zdjęć**. Walidacja i test mają po **500 zdjęć** i odpowiadają podziałowi z 2A/2B. Walidacja służy do wyboru wag i progu pewności. Test służy do końcowej oceny.

Uruchom w katalogu repozytorium w kernelu z PyTorch obsługującym GPU oraz z pakietami `ultralytics`, `opencv-python` i `ipykernel`. Na AMD z ROCm PyTorch używa nazwy urządzenia `cuda`. Brakujące pakiety w aktywnym kernelu można doinstalować poleceniem `%pip install ultralytics opencv-python`. Przy pierwszym treningu Ultralytics pobiera `yolov5su.pt`. Jeśli masz już plik z wagami, umieść go obok notebooka. Dane i wyniki zapisujemy w `cache/`, a źródłowe obrazy i etykiety odczytujemy z `100k/`.

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil

import numpy as np
import csv
from IPython.display import Markdown, display

SEED = 42
N_TRAIN = 15_000
IMG_SIZE = 960
BATCH = 8
EPOCHS = 30
DEVICE = 0

REPO = Path.cwd()
BDD = REPO / "100k"
CACHE = REPO / "cache" / "yolo_finetuning"
DATASET = CACHE / f"bdd_{N_TRAIN}_seed{SEED}"
RUN = CACHE / f"yolov5su_{N_TRAIN}_{IMG_SIZE}_{EPOCHS}ep_seed{SEED}"
CLASSES = [
    "traffic_sign", "traffic_light_red", "traffic_light_yellow",
    "traffic_light_green", "traffic_light_unknown",
]
COLOR_ID = {"red": 1, "yellow": 2, "green": 3, "none": 4}

## Sprawdzenie GPU i pakietów

In [ ]:
os.environ.setdefault("YOLO_CONFIG_DIR", str(CACHE / "ultralytics"))

# ROCm na Windows korzysta z ustawień sprawdzonych w poprzednim labie.
if os.name == "nt":
    os.environ.setdefault("MIOPEN_FIND_MODE", "FAST")
    for variable, directory in [
        ("MIOPEN_CUSTOM_CACHE_DIR", "miopen_kernels"),
        ("MIOPEN_USER_DB_PATH", "miopen_db"),
    ]:
        folder = CACHE / directory
        folder.mkdir(parents=True, exist_ok=True)
        os.environ.setdefault(variable, str(folder))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Wybierz kernel z PyTorch obsługującym GPU (ROCm lub CUDA).")

print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| ROCm:", torch.version.hip)
probe = torch.ones(4, device=f"cuda:{DEVICE}", requires_grad=True)
probe.square().sum().backward()
torch.cuda.synchronize(DEVICE)

try:
    from ultralytics import YOLO
    import cv2
except ModuleNotFoundError as error:
    raise RuntimeError(
        "W kernelu GPU uruchom %pip install ultralytics opencv-python, "
        "zrestartuj kernel i ponownie uruchom notebook."
    ) from error

best_weights = RUN / "weights" / "best.pt"
last_weights = RUN / "weights" / "last.pt"
finished = RUN / "training_complete"

if not finished.is_file() and not last_weights.is_file():
    model = YOLO(str(REPO / "yolov5su.pt"))


## 1. Podział i etykiety

Najpierw odtwarzamy losowanie z 2A/2B, żeby zachować dokładnie tę samą walidację i test. Rozszerzamy trening losowaniem z oficjalnego `train` po wykluczeniu walidacji. Do formatu YOLO zapisujemy środek i rozmiar ramki w skali od 0 do 1. Ramki bez pola pomijamy w eksporcie YOLO, a przy porównaniu z 2A/2B zachowujemy oryginalne ramki odniesienia.

In [ ]:
def label_ids(folder):
    return sorted(path.stem for path in folder.glob("*.json"))


official_train = label_ids(BDD / "train")
official_val = label_ids(BDD / "val")
if len(official_train) != 70_000 or len(official_val) != 10_000:
    raise RuntimeError("Oczekiwano pełnego BDD100K z JSON-ami obok obrazów w train i val.")

baseline_rng = random.Random(SEED)
baseline_pool = baseline_rng.sample(official_train, 2_500)
baseline_train, validation = baseline_pool[:2_000], baseline_pool[2_000:]
test = baseline_rng.sample(official_val, 500)

if (REPO / "test_images.json").is_file():
    previous_test = json.loads((REPO / "test_images.json").read_text(encoding="utf-8"))
    if sorted(test) != sorted(previous_test):
        raise RuntimeError("Lista zdjęć testowych różni się od testu 2A.")

excluded = set(baseline_pool)
extra_pool = [name for name in official_train if name not in excluded]
extra_train = random.Random(SEED).sample(extra_pool, N_TRAIN - len(baseline_train))
splits = {
    "train": ("train", sorted(baseline_train + extra_train)),
    "val": ("train", sorted(validation)),
    "test": ("val", sorted(test)),
}
assert sum(len(names) for _, names in splits.values()) == len(
    {name for _, names in splits.values() for name in names}
)
print({split: len(names) for split, (_, names) in splits.items()})

In [ ]:
def read_boxes(label_path):
    annotation = json.loads(label_path.read_text(encoding="utf-8"))
    boxes = []
    for obj in annotation["frames"][0].get("objects", []):
        if "box2d" not in obj:
            continue
        if obj["category"] == "traffic sign":
            class_id = 0
        elif obj["category"] == "traffic light":
            color = obj.get("attributes", {}).get("trafficLightColor", "none")
            class_id = COLOR_ID.get(color, 4)
        else:
            continue

        box = obj["box2d"]
        boxes.append((class_id, box["x1"], box["y1"], box["x2"], box["y2"]))
    return boxes


def clip_boxes(boxes, width, height):
    valid = []
    for class_id, x1, y1, x2, y2 in boxes:
        x1, x2 = np.clip([x1, x2], 0, width)
        y1, y2 = np.clip([y1, y2], 0, height)
        if x2 > x1 and y2 > y1:
            valid.append((class_id, x1, y1, x2, y2))
    return valid


def yolo_line(box, width, height):
    class_id, x1, y1, x2, y2 = box
    coordinates = (
        (x1 + x2) / (2 * width), (y1 + y2) / (2 * height),
        (x2 - x1) / width, (y2 - y1) / height,
    )
    return f"{class_id} " + " ".join(f"{value:.8f}" for value in coordinates)


def export_split(split, origin, names):
    images = DATASET / "images" / split
    labels = DATASET / "labels" / split
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)
    counts = np.zeros(len(CLASSES), dtype=int)
    truth = {}

    for name in names:
        source = BDD / origin / f"{name}.jpg"
        target = images / source.name
        image = cv2.imread(str(source))
        if image is None:
            raise RuntimeError(f"Nie udało się odczytać {source}.")
        height, width = image.shape[:2]
        truth[name] = read_boxes(source.with_suffix(".json"))
        boxes = clip_boxes(truth[name], width, height)
        if not target.exists():
            try:
                os.link(source, target)
            except OSError:
                shutil.copyfile(source, target)

        lines = [yolo_line(box, width, height) for box in boxes]
        (labels / f"{name}.txt").write_text("\n".join(lines), encoding="utf-8")
        for class_id, *_ in boxes:
            counts[class_id] += 1

    manifest = DATASET / f"{split}.txt"
    manifest.write_text(
        "\n".join((images / f"{name}.jpg").as_posix() for name in names) + "\n",
        encoding="utf-8",
    )
    return counts, truth

In [ ]:
counts_by_split, reference = {}, {}
for split, (origin, names) in splits.items():
    print(f"Eksport {split}: {len(names)} obrazów", flush=True)
    counts_by_split[split], reference[split] = export_split(split, origin, names)

data_config = {"path": DATASET.as_posix(), "names": dict(enumerate(CLASSES))}
data_config.update({split: f"{split}.txt" for split in splits})
data_yaml = DATASET / "data.yaml"
data_yaml.write_text(json.dumps(data_config, indent=2), encoding="utf-8")
def show_table(headers, rows, path=None):
    if path is not None:
        with path.open("w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(headers)
            writer.writerows(rows)

    def format_row(row):
        return "| " + " | ".join(str(value) for value in row) + " |"

    lines = [format_row(headers), format_row(["---"] * len(headers))]
    lines.extend(format_row(row) for row in rows)
    display(Markdown("\n".join(lines)))


show_table(["Klasa", *splits], [
    [name, *(counts[i] for counts in counts_by_split.values())]
    for i, name in enumerate(CLASSES)
])

## 2. Fine-tuning na GPU

Uczymy z obrazami o dłuższym boku 960 px i budżetem 30 epok. Format prostokątny ogranicza dopełnienie obrazu. Wyłączamy zmianę odcienia, żeby augmentacja zachowała etykietę koloru. Na ROCm zaczynamy od FP32. Jeśli komórka treningowa została już wykonana do końca, używa zapisanych wag.

In [ ]:
if not finished.is_file():
    if last_weights.is_file():
        model = YOLO(str(last_weights))
        model.train(resume=True, device=DEVICE, workers=0, batch=4)
    else:
        model.train(
            data=str(data_yaml),
            device=DEVICE,
            imgsz=IMG_SIZE,
            batch=BATCH,
            epochs=EPOCHS,
            patience=10,
            seed=SEED,
            deterministic=False,
            workers=0,
            amp=False,
            rect=True,
            mosaic=0.0,
            hsv_h=0.0,
            project=str(RUN.parent),
            name=RUN.name,
            exist_ok=True,
        )
    if not best_weights.is_file():
        raise FileNotFoundError(best_weights)
    finished.touch()

model = YOLO(str(best_weights))
print(best_weights)

## 3. Ocena na wspólnym teście

Dobieramy próg confidence na walidacji według średniego F1 dla znaków i lokalizacji sygnalizatorów. Przy ocenie lokalizacji wszystkie kolory traktujemy jak jedną klasę sygnalizatora. Kolor oceniamy na dopasowanych ramkach, a AP per klasa pokazuje łącznie poprawność lokalizacji i koloru. Stosujemy dopasowanie jeden do jednego przy IoU ≥ 0,5, jak w 2A/2B.

In [ ]:
CONF_GRID = [0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]


def predict_split(split):
    _, names = splits[split]
    manifest = DATASET / f"{split}.txt"
    predictions = {}
    results = model.predict(
        str(manifest), stream=True, batch=BATCH, conf=0.01, imgsz=IMG_SIZE,
        device=DEVICE, verbose=False,
    )
    for result in results:
        name = Path(result.path).stem
        boxes = result.boxes
        predictions[name] = (
            boxes.xyxy.cpu().numpy(),
            boxes.conf.cpu().numpy(),
            boxes.cls.cpu().numpy().astype(int),
        )
    if predictions.keys() != set(names):
        raise RuntimeError(f"Brakuje predykcji dla splitu {split}.")
    return predictions


def iou(box, candidates):
    intersection_size = np.maximum(
        np.minimum(box[2:], candidates[:, 2:]) -
        np.maximum(box[:2], candidates[:, :2]), 0,
    )
    intersection = intersection_size.prod(axis=1)
    box_area = np.prod(box[2:] - box[:2])
    candidate_area = (candidates[:, 2:] - candidates[:, :2]).prod(axis=1)
    return intersection / np.maximum(box_area + candidate_area - intersection, 1e-9)


def evaluate(split, predictions, threshold, signs):
    tp = fp = fn = 0
    colors = np.zeros((4, 4), dtype=int)
    for name, truth in reference[split].items():
        gt = np.asarray([row for row in truth if (row[0] == 0) == signs]).reshape(-1, 5)
        boxes, scores, classes = predictions[name]
        keep = np.flatnonzero((scores >= threshold) & ((classes == 0) == signs))
        order = keep[np.argsort(-scores[keep])]
        used = np.zeros(len(gt), dtype=bool)

        for index in order:
            available = np.flatnonzero(~used)
            if len(available) == 0:
                break
            overlaps = iou(boxes[index], gt[available, 1:])
            best = overlaps.argmax()
            if overlaps[best] >= 0.5:
                match = available[best]
                used[match] = True
                if not signs:
                    colors[int(gt[match, 0]) - 1, classes[index] - 1] += 1

        matches = int(used.sum())
        tp += matches
        fp += len(keep) - matches
        fn += len(gt) - matches

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * tp / max(2 * tp + fp + fn, 1)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall, "F1": f1}, colors

In [ ]:
val_predictions = predict_split("val")
scores = {}
for threshold in CONF_GRID:
    signs, _ = evaluate("val", val_predictions, threshold, signs=True)
    lights, _ = evaluate("val", val_predictions, threshold, signs=False)
    scores[threshold] = (signs["F1"] + lights["F1"]) / 2

threshold = max(scores, key=scores.get)
print(f"Próg confidence wybrany na walidacji: {threshold:g}")

test_predictions = predict_split("test")
sign_metrics, _ = evaluate("test", test_predictions, threshold, signs=True)
light_metrics, confusion = evaluate("test", test_predictions, threshold, signs=False)
show_table(["Obiekty", *sign_metrics], [
    [name, *(round(value, 4) for value in metrics.values())]
    for name, metrics in [("znaki drogowe", sign_metrics), ("sygnalizatory", light_metrics)]
], RUN / "test_detection.csv")
print("F1 sygnalizatorów z zapisanych wyników: 2A = 0.270, hybryda 2B = 0.387")

color_names = ["red", "yellow", "green", "unknown"]
show_table(["Etykieta / predykcja", *color_names], [
    [name, *row] for name, row in zip(color_names, confusion)
], RUN / "test_color_confusion.csv")
print(f"Accuracy koloru na trafionych ramkach: {confusion.trace() / max(confusion.sum(), 1):.3f}")

In [ ]:
metrics = model.val(
    data=str(data_yaml), split="test", device=DEVICE, imgsz=IMG_SIZE,
    batch=BATCH, workers=0, plots=True, project=str(RUN), name="test", exist_ok=True,
)
class_ap = [
    [CLASSES[int(class_id)], round(float(metrics.box.ap50[i]), 4), round(float(metrics.box.ap[i]), 4)]
    for i, class_id in enumerate(metrics.box.ap_class_index)
]
show_table(["Klasa", "AP50", "AP50-95"], class_ap, RUN / "test_ap.csv")